### AOI & Hexes

In [1]:
from accessx.aoi import load_aoi, make_hex_grid

bbox_milano = (
    9.1860,   # minx (lon)
    45.4620,  # miny (lat)
    9.1960,   # maxx (lon)
    45.4680,  # maxy (lat)
)
city_epsg = 32632
buffer=100


aoi = load_aoi(bbox=bbox_milano, buffer_m=buffer, utm_crs=city_epsg)
hexes = make_hex_grid(aoi,resolution=10, clip=True)

### Get street network

In [ ]:
from accessx.graph import build_network, save_graph

city_epsg = 32632
buffer=100

G_proj = build_network(
    AOI=aoi,
    city_epsg=city_epsg,
    buffer_m=buffer,
    network_type="walk",
    simplify=False,
    retain_all=True,
)


save_graph(
    G_proj,
    out_dir= "data/street_network/",
    base_name="original_graph",
    save_nodes=True,
    save_edges=True,
)

### Calculate cost per street segment 

The cost can be the time to traverse a street based on a defined constant speed...

In [3]:
from accessx.cost import add_time_cost_constant_speed

# time-cost based on costant speed defined by the user
G_proj = add_time_cost_constant_speed(G_proj, speed_kmh=4.5, cost_col="avg_time")


...or a the time when using speed adjusted to the slope of the streets (in %, assuming slope_pct has been added to the graph)...

In [4]:
from accessx.cost import add_slope_based_time, add_edge_cost

# custom cost defined by specific function that receives edge as input and returns cost
# example with function that returns time-cost based on slope-related speeds

G_proj = add_edge_cost(
    G_proj,
    cost_fn=add_slope_based_time(slope_col="slope_pct"),
    cost_col="slope_based_time",
)


... or your own cost function..

In [5]:
def discomfort(edge):
    bad = 1.0 if edge.get("lit") == "yes" else 1.5
    return edge["length"] * bad

G_proj = add_edge_cost(G_proj, cost_fn=discomfort, cost_col="discomfort")

In [ ]:
save_graph(
    G_proj,
    out_dir= "data/street_network/",
    base_name="graph_with_cost",
    save_nodes=False,
    save_edges=True,
)

city_epsg = 32632

G_proj = load_graph(out_dir="data/street_network/", base_name="graph_with_cost", crs=city_epsg)
